# Response API Tool Calls

* [/responses](https://docs.litellm.ai/docs/response_api)

> LiteLLM provides an endpoint in the spec of [OpenAI's /responses API](https://docs.litellm.ai/docs/response_api).


* [LiteLLM - Getting Started](https://docs.litellm.ai/docs/)
* [LiteLLM Cookbook](https://github.com/BerriAI/litellm/tree/main/cookbook)

# Setup

In [1]:
%%html
<style>
table {float:left}
</style>

In [2]:
import os
import json
import operator
from datetime import datetime, timedelta, timezone
from typing import (
    List, Dict, Any, Literal, Optional, Callable, Annotated
)

import regex as re
from pydantic import BaseModel, Field, ConfigDict

from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END
import litellm
import mdformat
import trafilatura
from IPython.display import Markdown, display

## API Keys

In [3]:
path_to_openai_key:str = os.path.expanduser('~/.openai/api_key')
with open(path_to_openai_key, 'r', encoding='utf-8') as file:
    os.environ["OPENAI_API_KEY"] = file.read().strip()

path_to_tavily_key:str = os.path.expanduser('~/.tavily/api_key')
with open(path_to_tavily_key, 'r', encoding='utf-8') as file:
    os.environ["TAVILY_API_KEY"] = file.read().strip()

## Models

In [4]:
MODEL: str = "openai/gpt-5.2"   # LiteLLM format: "<provider>/<model-name>"

---
# Tool Calling Protocol


## OpenAI API Specification


OpenAI defined the **Function (Tool) Calling Protocol** as in [Function calling](https://developers.openai.com/api/docs/guides/function-calling). 

  1. LLM responds with role: "assistant" message containing tool_calls (with id, function.name, function.arguments)
  2. You execute the tools                                                      
  3. You send back one message per tool call with role: "tool", tool_call_id matching the original id, and content containing the result as a string

There are six stages in the Protocol.

1. Tool Definition by Application and Declaration to LLM at Prompt Message
> When we make an API request to the model with a prompt, we can include a list of tools the model could consider using.

2. Tool Call Decision by LLM  (**Reasoing** and **Decisoning**)
> tool call refers to a special kind of decision response from LLM to call one of the tools to execute the prompt given.
   
3. Tool Execution by Application
4. Tool Output Usage by LLM
> tool call output refers to the response a tool generates using the input from a model’s tool call. We send **all of the tool definition, the original prompt, the model’s tool call, and the tool call output back** to LLM to finally receive a text response.

5. LLM completes the prompt

### Tool Calling Workflow

```
1. Application to LLM
   │ Define and Declare Tools in the prompt
   ▼
2. LLM
   │ Reason/Decide Tool Calls
   ▼
3. Application/Orchestration
   │ Tool Executions
   ▼
4. LLM Receives Tool Outputs
   │ Use Tool Outputs
   ▼
5. LLM Complete the prompt
```

<img src="../image/tool_call_flow.png" align="left" width=500/>



## LiteLLM

Use LiteLLM as the wrapper for the OpenAI Response API for Function calling.

* [LiteLLM - Response API](https://docs.litellm.ai/docs/providers/openai/responses_api)
* [LiteLLM - Function Calling](https://docs.litellm.ai/docs/completion/function_call)

1. `litellm.responses()` with `tools=` and `instructions=`
2. Parse `response.output` for `type=="function_call"` items and execute functions
3. Second `litellm.responses()` call with tool results appended to `input`


## Tool Definition

* [Defining Functions (Tools)](https://developers.openai.com/api/docs/guides/function-calling#defining-functions)

First, you define a tool based on the OpenAI definition. This is what LiteLLM accepts as well.

| Field       | Description                                          |
|-------------|------------------------------------------------------|
| type        | This should always be function                       |
| name        | The function’s name (e.g. get_weather)               |
| description | Details on when and how to use the function          |
| parameters  | JSON schema defining the function’s input arguments  |
| strict      | Whether to enforce strict mode for the function call |


### Tool Definition Format

The tool definition format differs between Chat Completion and Response API:

**Response API** — flat, no nested `"function"` wrapper:
```
{
  "type": "function",
  "name": "...",             ← promoted to top level
  "description": "...",
  "parameters": { JSON schema }
}
```

* [Response API - Function Calling](https://developers.openai.com/api/docs/guides/function-calling)

**Example (Response API):**
```
{
  "type": "function",
  "name": "get_weather",
  "description": "Retrieves current weather for the given location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "City and country e.g. Bogota, Colombia"
      },
      "units": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Units the temperature will be returned in."
      }
    },
    "required": ["location", "units"],
    "additionalProperties": false
  },
  "strict": true
}
```

### Namespace

* [Defining namespace](https://developers.openai.com/api/docs/guides/function-calling#defining-namespaces)

> Namespaces help organize similar tools and are especially useful when the model must choose between tools that serve different systems or purposes.

```
{
  "type": "namespace",
  "name": "crm",
  "description": "CRM tools for customer lookup and order management.",
  "tools": [
      tool_definition+
 ]
}
```


### Tool Parameter JSON Schema

[Pydantic JSON Schema](https://docs.pydantic.dev/latest/concepts/json_schema/) to generate the JSON Schema for the Tool Definition. See [Tavilty Search](https://docs.tavily.com/sdk/python/reference#tavily-search) for the tool parameters.


#### Tool 

In [5]:
# Function for the Search Tool
search_tool: Callable = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
if not hasattr(search_tool, "tool_name"):
    search_tool.name = "search_tool"

name_to_tool: Dict[str, Callable] = {
    # search_tool.name: search_tool.search
    search_tool.name: search_tool.search
}

#### Tool Parameters

Use Pydantic to generate the JSON Schema for the parameters. See [tavily search](https://docs.tavily.com/sdk/python/reference#tavily-search) for the prameter details.



In [6]:
# Parameters for the Search Tool (Tavily)
# https://docs.tavily.com/sdk/python/reference#tavily-search
class SearchToolParameters(BaseModel):
    """
    Search the web for current events, news, or deep research.
    """
    query: str = Field(description="The search query to look up")
    
    topic: Literal["general", "news", "finance"] = Field(
        default="general",
        description="Category of search. Use 'news' for current events/politics, 'finance' for market data."
    )
    search_depth: Literal["basic", "advanced"] = Field(
        default="basic",
        description="Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context."
    )
    time_range: Optional[Literal["day", "week", "month", "year"]] = Field(
        default=None,
        description="Filter results by publication date. Especially useful with topic='news'."
    )
    max_results: int = Field(
        default=5, ge=1, le=10,
        description="Number of search results to return."
    )
    model_config = ConfigDict(extra="forbid")  # adds additionalProperties: false

In [7]:
search_tool_parameters: Dict[str, Any] = SearchToolParameters.model_json_schema()
print(json.dumps(search_tool_parameters, indent=2, default=str))

{
  "additionalProperties": false,
  "description": "Search the web for current events, news, or deep research.",
  "properties": {
    "query": {
      "description": "The search query to look up",
      "title": "Query",
      "type": "string"
    },
    "topic": {
      "default": "general",
      "description": "Category of search. Use 'news' for current events/politics, 'finance' for market data.",
      "enum": [
        "general",
        "news",
        "finance"
      ],
      "title": "Topic",
      "type": "string"
    },
    "search_depth": {
      "default": "basic",
      "description": "Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context.",
      "enum": [
        "basic",
        "advanced"
      ],
      "title": "Search Depth",
      "type": "string"
    },
    "time_range": {
      "anyOf": [
        {
          "enum": [
            "day",
            "week",
            "month",
            "year"
          ],
          "type": "str

#### Tool Description

In [8]:
search_tool_description: str = SearchToolParameters.__doc__.strip()
print(search_tool_description)

Search the web for current events, news, or deep research.


### Tool Definition of the Tabiliy Web Search

In [9]:
# Response API: flat tool definition — name/description/parameters at top level (no "function" wrapper)
# Ref: https://docs.litellm.ai/docs/response_api
#      https://developers.openai.com/api/docs/guides/function-calling
search_tool_definition = {
    "type": "function",
    "name": search_tool.name,
    "description": search_tool_description,
    "parameters": search_tool_parameters,
    # "strict": True causes BadRequestError when optional fields are not in "required".
    # Pydantic model_json_schema() only adds fields with no default to "required".
    # Fix: either omit strict or mark all fields as required in the schema.
    # BadRequestError: Error code: 400 - {'error': {
    #   'message': "Invalid schema for function 'search_tool': In context=(),
    #              'required' is required to be supplied and to be an array
    #              including every key in properties. Missing 'topic'.",
    #   'param': 'tools[0].parameters',
    #   'code': 'invalid_function_parameters'
    # }}
    # "strict": True
}
print(json.dumps(search_tool_definition, indent=2, default=str))

{
  "type": "function",
  "name": "search_tool",
  "description": "Search the web for current events, news, or deep research.",
  "parameters": {
    "additionalProperties": false,
    "description": "Search the web for current events, news, or deep research.",
    "properties": {
      "query": {
        "description": "The search query to look up",
        "title": "Query",
        "type": "string"
      },
      "topic": {
        "default": "general",
        "description": "Category of search. Use 'news' for current events/politics, 'finance' for market data.",
        "enum": [
          "general",
          "news",
          "finance"
        ],
        "title": "Topic",
        "type": "string"
      },
      "search_depth": {
        "default": "basic",
        "description": "Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context.",
        "enum": [
          "basic",
          "advanced"
        ],
        "title": "Search Depth",
        "type

## Tool Declaration

You declare the **tool definitions** to the LLM.

**Response API** — flat format:
```
tools = [
    {
      "type": "function",
      "name": "SearchTool",
      "description": "...",
      "parameters": { JSON Schema }
    },
    ...
]
```

**Chat Completion (deprecated)** — nested `"function"` wrapper:
```
tools = [
    {
      "type": "function",
      "function": {
        "name": "SearchTool",
        "description": "...",
        "parameters": { JSON Schema }
      }
    },
    ...
]
```

* [LiteLLM - Response API](https://docs.litellm.ai/docs/providers/openai/responses_api)

```python
# Response API
response = litellm.responses(
    model=model,
    input=input,
    tools=tools,            # <- declare tools here (flat format)
    instructions=instructions,
)
```

*Example input:*
```
{
  "model": "openai/gpt-4o",
  "input": [
    {
        "role": "user",
        "content": "Weather in Tokyo?"
    }
  ],
  "tools": [
    {
      "type": "function",
      "name": "get_weather",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {"type": "string"}
        }
      }
    }
  ]
}
```


### Tool Declartion for the Search

In [10]:
tool_declarations: List[Dict[str, Any]] = [
    search_tool_definition
]
tool_declarations

[{'type': 'function',
  'name': 'search_tool',
  'description': 'Search the web for current events, news, or deep research.',
  'parameters': {'additionalProperties': False,
   'description': 'Search the web for current events, news, or deep research.',
   'properties': {'query': {'description': 'The search query to look up',
     'title': 'Query',
     'type': 'string'},
    'topic': {'default': 'general',
     'description': "Category of search. Use 'news' for current events/politics, 'finance' for market data.",
     'enum': ['general', 'news', 'finance'],
     'title': 'Topic',
     'type': 'string'},
    'search_depth': {'default': 'basic',
     'description': "Use 'basic' for quick facts. Use 'advanced' for complex queries needing more context.",
     'enum': ['basic', 'advanced'],
     'title': 'Search Depth',
     'type': 'string'},
    'time_range': {'anyOf': [{'enum': ['day', 'week', 'month', 'year'],
       'type': 'string'},
      {'type': 'null'}],
     'default': None,
  

## LLM
### Input to LLM

Build the input items to LLM (the prompt).

* Response API: `input=` takes a list of message dicts `{role, content}`
* System instructions are passed separately as `instructions=`


In [11]:
# Response API: system instructions go in instructions= (separate from input items)
# litellm.responses(instructions=..., input=[user/assistant turns only])
# Ref: https://docs.litellm.ai/docs/response_api
#      https://developers.openai.com/api/docs/guides/migrate-to-responses#system-and-developer-messages
SYSTEM_INSTRUCTIONS: str = """
You are a helpful assistant. Use tools when needed.
Verify the relevance of retrieved results before using them, and apply an intelligent
intent filter so you keep only the news items that align with the user inquiry.
Only report items that are relevant to the user request and supported by the tool output.
If the search results are noisy or insufficient, say that and do not infer missing facts.
Respnse follows the given Python JSON schema where:
1. news field is the news articies as list.
2. uri field is the URI of the news article found.
"""

# Response API: input= carries only user/assistant turn items — no system message
input_prompt: List[Dict[str, str]] = [
    {
        "role": "user",
        "content": "What are the top news headlines about Iran US conflicts?"
    }
]

### Output format from LLM


In [12]:
# RFC 3986 URI: scheme "://" followed by non-whitespace characters                                                                                                                                        
UriStr = Annotated[str, Field(pattern=r"^[a-zA-Z][a-zA-Z0-9+\-.]*://\S+$")]

class ResponseStructuredFormat(BaseModel):
    """Schema for the Python query answer"""
    news: list[str] = Field(
        description="List of news"
    )
    uri: list[UriStr] = Field(
        description="URIs of the specifications referred"
    )
    #--------------------------------------------------------------------------------
    # JSON Schema additionalProperty: False is required for the API to work
    # community.openai.com/t/schema-additionalproperties-must-be-false-when-strict-is-true/929996
    #--------------------------------------------------------------------------------
    # BadRequestError: Error code: 400 - {
    # {
    #   "message": "Invalid schema for response_format ... additionalProperties is required to be supplied and to be false.",
    #   "type": "invalid_request_error",
    #   "param": "text.format.schema",
    #   "code": "invalid_json_schema"
    # }
    #--------------------------------------------------------------------------------
    model_config = ConfigDict(extra="forbid")  # adds additionalProperties: false

ResponseStructuredFormat.model_json_schema()

{'additionalProperties': False,
 'description': 'Schema for the Python query answer',
 'properties': {'news': {'description': 'List of news',
   'items': {'type': 'string'},
   'title': 'News',
   'type': 'array'},
  'uri': {'description': 'URIs of the specifications referred',
   'items': {'pattern': '^[a-zA-Z][a-zA-Z0-9+\\-.]*://\\S+$',
    'type': 'string'},
   'title': 'Uri',
   'type': 'array'}},
 'required': ['news', 'uri'],
 'title': 'ResponseStructuredFormat',
 'type': 'object'}

### LLM Call

* [Function Calling](https://developers.openai.com/api/docs/guides/tools?tool-type=function-calling)

In [13]:
# litellm.responses() mirrors the OpenAI Responses API — no client object needed.
# Ref: https://docs.litellm.ai/docs/response_api
response = litellm.responses(
    model=MODEL,
    instructions=SYSTEM_INSTRUCTIONS,
    input=input_prompt,   # plain text string or list of turn dicts
    tools=tool_declarations,
    tool_choice="auto",
    temperature=0,
    max_output_tokens=2048,
    text={
        "format": {
            # Structured Output: force the model to emit valid JSON matching the schema.
            # Ref: https://docs.litellm.ai/docs/response_api#structured-outputs
            "type": "json_schema",
            "name": "ResponseStructuredFormat",
            "strict": True,
            "schema": ResponseStructuredFormat.model_json_schema()
        }
    },
    stream=False
)

In [14]:
# litellm response is a dict-like object — use json.dumps instead of .model_dump_json()
print(json.dumps(response, indent=4, default=str))

"id='resp_bGl0ZWxsbTpjdXN0b21fbGxtX3Byb3ZpZGVyOm9wZW5haTttb2RlbF9pZDpOb25lO3Jlc3BvbnNlX2lkOnJlc3BfMDgzZjJlZTlmZTA2Yzc0NDAwNjlkMzI3YmQ2ODM4ODFhMzg5NmE3NGQ5MDZjM2UwM2M=' created_at=1775445949 error=None incomplete_details=None instructions='\\nYou are a helpful assistant. Use tools when needed.\\nVerify the relevance of retrieved results before using them, and apply an intelligent\\nintent filter so you keep only the news items that align with the user inquiry.\\nOnly report items that are relevant to the user request and supported by the tool output.\\nIf the search results are noisy or insufficient, say that and do not infer missing facts.\\nRespnse follows the given Python JSON schema where:\\n1. news field is the news articies as list.\\n2. uri field is the URI of the news article found.\\n' metadata={} model='gpt-5.2-2025-12-11' object='response' output=[ResponseFunctionToolCall(arguments='{\"query\":\"Iran US conflict top headlines\",\"topic\":\"news\",\"search_depth\":\"basic\",\"

## LLM Tool Calls

### Tool call handling logic

* [OpenAI - Handling function calls](https://developers.openai.com/api/docs/guides/function-calling#handling-function-calls)

**Response API** — tool calls are Items in `response.output` with `type == "function_call"`:

```python
for item in response.output:
    if item.type != "function_call":
        continue
    name = item.name
    args = json.loads(item.arguments)
    result = call_function(name, args)
    input_items.append({
        "type": "function_call_output",  # <- NOT role="tool"
        "call_id": item.id,              # <- NOT tool_call_id
        "output": str(result)            # <- NOT content
    })
```

### Tool Calls

**Response API**: the tool call is a top-level Item in `response.output`:

```
{
  "type": "function_call",
  "id": "call_JVHGxhAhxbmAlwiDkuc7fXk6",    <- call_id (used in function_call_output)
  "name": "search_tool",
  "arguments": "{\"query\":\"AI news\",\"topic\":\"news\",\"search_depth\":\"basic\"}"
}
```

In [15]:
# Verify litellm.responses() returned at least one output item
assert response.output, \
    f"Expected non-empty output:\n{json.dumps(response, indent=2, default=str)}" 

In [16]:
def extract_tool_calls(response) -> List[Any]:
    """Extract function_call items from a litellm.responses() response.

    Response API: tool calls appear as items in response.output with type == "function_call".
      Each item has: .type, .id, .call_id, .name, .arguments (JSON string)
      Ref: https://docs.litellm.ai/docs/response_api
           https://developers.openai.com/api/docs/guides/function-calling#handling-function-calls

    Compare with Chat Completion (deprecated):
      response.choices[0].message.tool_calls
      Each: .id, .function.name, .function.arguments
    """
    return [item for item in response.output if item.type == "function_call"]

In [17]:
def in_time_window(result: Dict[str, Any], query: str, now: datetime) -> bool:
    """Return True when the result was published within the last month."""
    del query  # Reserved for query-aware time parsing later.
    published_date = result.get("published_date")
    if not published_date:
        return False
    try:
        published_at = datetime.strptime(
            published_date, "%a, %d %b %Y %H:%M:%S %Z"
        ).replace(tzinfo=timezone.utc)
    except ValueError:
        return False
    return published_at >= (now - timedelta(days=30))

def is_bad_page_type(result: Dict[str, Any]) -> bool:
    """Return True for page types that are poor RAG evidence for this tutorial."""
    negative_markers = ('/event/', 'event', 'retreat', 'viewership', '/journalism/')
    title = str(result.get("title", "")).lower()
    url = str(result.get("url", "")).lower()
    return any(marker in title or marker in url for marker in negative_markers)

def keep_result(query: str, result: Dict[str, Any], now: datetime, threshold: float=0.70) -> bool:
    """Return True when a Tavily result is usable for the final answer."""
    print(json.dumps(result, indent=4, default=str))
    if not in_time_window(result, query, now):
        print("did not match time window")
        return False
    if is_bad_page_type(result):
        print("bad page type")
        return False
    if float(result.get("score", 0.0)) < threshold:
        print(f"no confidence < {threshold}")
        return False

    print("found")
    return True

def filter_search_results(results: List[Dict[str, Any]], query: str) -> List[Dict[str, Any]]:
    """Keep only search hits that are recent, relevant, and usable."""
    now = datetime.now(timezone.utc)
    return [result for result in results if keep_result(query, result, now)]

def format_search_result(result: Dict[str, Any]) -> str:
    """Serialize one Tavily result as JSON for the tool response."""
    payload = {
        "title": result.get("title", ""),
        "url": result.get("url", ""),
        "content": result.get("content", ""),
        "published_date": result.get("published_date"),
    }
    return json.dumps(payload, ensure_ascii=False)

def execute_tool_calls(tool_calls: List[Any]) -> List[Dict[str, Any]]:
    """Execute function_call items and return function_call_output items.

    Args:
        tool_calls: function_call items from response.output.
                    Each has: .type, .call_id, .name, .arguments (JSON string)
    Returns:
        List of function_call_output dicts to append to the next litellm.responses() input:
        [{"type": "function_call_output", "call_id": ..., "output": ...}]

    Response API tool result format:
        {"type": "function_call_output", "call_id": ..., "output": <str>}
    Compare with Chat Completion (deprecated):
        {"role": "tool", "tool_call_id": ..., "name": ..., "content": <str>}

    Ref: https://docs.litellm.ai/docs/response_api
         https://developers.openai.com/api/docs/guides/function-calling#formatting-results
    """
    _tool_outputs: List[Dict[str, Any]] = []
    for tool_call in tool_calls:
        if tool_call.type != "function_call":
            continue

        func_name: str = tool_call.name                               # Chat Completion: tool_call.function.name
        func_args: Dict[str, Any] = json.loads(tool_call.arguments)   # Chat Completion: tool_call.function.arguments
        execution: Any = name_to_tool[func_name](**func_args)
        results = execution.get("results", [])
        if func_name == "search_tool":
            results = filter_search_results(results=results, query=func_args.get("query", ""))
        if not results:
            content = json.dumps({
                "query": func_args.get("query", ""),
                "results": [],
                "note": "No clearly relevant search results were found. Ask for a narrower query or different source."
            }, ensure_ascii=False)
        else:
            content = json.dumps({
                "query": func_args.get("query", ""),
                "results": [json.loads(format_search_result(r)) for r in results]
            }, ensure_ascii=False)

        _tool_outputs.append({
            "type": "function_call_output",  # Response API field (Chat Completion used role="tool")
            "call_id": tool_call.call_id,    # call_id links back to the function_call item
            # tool_call.id     = output item ID  (e.g. "fc_077c...")  — do NOT use this
            # tool_call.call_id = tool call ref  (e.g. "call_abc...") — use this
            "output": content,               # Response API field (Chat Completion used "content")
        })

    return _tool_outputs

In [18]:
import markdown
from html.parser import HTMLParser

class _StripHTML(HTMLParser):
    """Extract plain text from HTML, used as fallback in render_content."""
    def __init__(self):
        super().__init__()
        self._parts = []
    def handle_data(self, data):
        self._parts.append(data)
    def get_text(self) -> str:
        return ''.join(self._parts)

def render_content(content: str, url: str) -> str:
    """Return clean plain text from url for display(Markdown(...)).
    Fetches original article text via trafilatura (preferred).
    Falls back to parsing the Tavily markdown snippet via the markdown
    package to strip heading/formatting artifacts.
    Args:
        content: Tavily markdown snippet (fallback source).
        url:     Original article URL (preferred source).
    """
    if url:
        downloaded = trafilatura.fetch_url(url)
        fetched = trafilatura.extract(downloaded) if downloaded else None
        if fetched:
            return fetched.replace('\xa0', ' ')
    # Fallback: markdown → HTML → strip tags → plain text
    html = markdown.markdown(content)
    extractor = _StripHTML()
    extractor.feed(html)
    return extractor.get_text().replace('\xa0', ' ')

In [19]:
def show_tool_executions(outputs: List[Dict[str, Any]]) -> None:
    """Display tool execution results.

    Parses JSON tool payloads and calls render_content(content, url)
    to fetch clean article text for display.
    """
    for msg in outputs:
        # Response API tool result: key is "output"
        # {"type": "function_call_output", "call_id": ..., "output": "..."}
        payload = json.loads(msg.get("output", "{}"))
        results = payload.get("results", [])
        if not results:
            note = payload.get("note", "No tool results to display.")
            display(Markdown(note))
            continue
        for result in results:
            source = result.get("title", "")
            url = result.get("url", "")
            article_content = result.get("content", "")
            published_date = result.get("published_date")
            clean = render_content(content=article_content, url=url)
            header = f"**{source}**"
            if published_date:
                header += f"\n\nPublished: {published_date}"
            display(Markdown(f"{header}\n\n{clean}"))
            display(Markdown("---"))

In [20]:
tool_calls = extract_tool_calls(response=response)
tool_outputs = execute_tool_calls(tool_calls=tool_calls)

{
    "url": "https://www.yahoo.com/news/articles/iran-remains-stubborn-foe-absorbing-154218161.html",
    "title": "Iran remains a stubborn foe after absorbing massive US-Israeli attacks - yahoo.com",
    "score": 0.99448806,
    "published_date": "Tue, 31 Mar 2026 15:42:18 GMT",
    "content": "## Top Stories:. * Trump to address nation on war. * New front in Iran war. Add Yahoo as a preferred source to see more of our stories on Google. BEIRUT (AP) \u2014 Since the United States and Israel launched their war against Iran on Feb. 28, the Trump administration claims to have all but \"obliterated\" the Islamic Republic's military capabilities. Its cheap drones slip through its neighbors\u2019 air defenses, shattering Gulf Arab nations\u2019 carefully curated images of invincibility and wounding U.S. troops. ## Iran is firing fewer ballistic missiles than at start of the war. Over a month into the war, Trump administration officials continue to refer to the first 72 hours as their point

In [21]:
show_tool_executions(outputs=tool_outputs)

**Iran remains a stubborn foe after absorbing massive US-Israeli attacks - yahoo.com**

Published: Tue, 31 Mar 2026 15:42:18 GMT

Iran remains a stubborn foe after absorbing massive US-Israeli attacks
BEIRUT (AP) — Since the United States and Israel launched their war against Iran on Feb. 28, the Trump administration claims to have all but "obliterated" the Islamic Republic's military capabilities. Defense Secretary Pete Hegseth declared last week that “never in recorded history has a nation’s military been so quickly and so effectively neutralized.”
But after more than a month of punishing U.S.-Israeli airstrikes, a degraded Iranian military nonetheless remains a stubborn foe. Its steady stream of strikes against Israel and Gulf Arab neighbors are causing regional chaos and an outsized economic and political shock.
Its missiles continue to penetrate Israeli airspace and kill civilians. Its cheap drones slip through its neighbors’ air defenses, shattering Gulf Arab nations’ carefully curated images of invincibility and wounding U.S. troops. Its threats to attack oil and gas tankers strangle the Strait of Hormuz, sending energy prices soaring.
U.S. President Donald Trump has sought negotiations and threatened extreme destruction in hopes of securing Iran’s stockpile of enriched uranium and compelling it to reopen the Strait of Hormuz. To maintain its leverage, Iran just needs to withstand the conflict long enough to pressure Washington to seek an off-ramp, experts say.
“Their strategy is to try to cause sustained pain and to drive up the costs of the war for the U.S.,” said Kelly Grieco, an expert in U.S. military strategy and operations who is a senior fellow at the Washington-based Stimson Center think tank.
Iran is firing fewer ballistic missiles than at start of the war
Since the first day of the U.S.-Israeli bombing campaign, officials from both countries have repeatedly pointed to a steep drop-off in Iran’s firing of ballistic missiles as proof that their efforts to destroy launchers and weapons stockpiles were working.
Chairman of the Joint Chiefs of Staff Gen. Dan Caine told reporters on March 4 that Iran’s “ballistic missile shots fired are down 86% from the first day of fighting and their one-way attack drone shots are down 73%." At a press briefing two weeks later, Hegseth said the volume of Iran’s ballistic missile attacks had dropped "90% since the start of the conflict.”
On Tuesday, Hegseth told reporters at the Pentagon that in the past 24 hours Iran had fired its “lowest number” of missiles and drones, though neither he nor Caine gave any updated percentages. Trump said Tuesday on Truth Social that “Iran has been, essentially, decimated.”
Claims of a slowdown in Iranian strikes are backed up by independent data from Armed Conflict Location & Event Data (ACLED), a U.S.-based group that tracks conflicts around the world.
On March 1, the second day of the war, Iran fired off almost 100 strikes. The next day, its strike count dropped to 53 and it hovered at that rate for the next few days. In the three and a half weeks since March 6, ACLED data shows Iran hasn’t fired more than 50 strikes on any given day. A “strike,” in ACLED’s methodology, can include multiple individual strikes in the same location on the same day.
Iran has maintained an average of 30 strikes each day for the last three weeks, and at various points it has picked up its tempo of attacks.
“That makes me question whether it’s a capacity issue or a strategy issue,” Grieco said of the initial decline in Iran's strike rate. In other words, Iran may not be running out of firepower as much as deliberately rationing its missiles and drones.
Iran fires more drones that are harder to intercept
The ACLED data shows that some 40% of Iran's salvos across the region are breaking through air defenses, signaling strain on American and Israeli supplies of interceptors. Iran has been deploying fewer missiles but more low-flying drones that are harder to intercept.
More in World
“We are vaporizing billions of dollars in long-range anti-missile defenses, which are scarce national resources,” said Tom Karako, the director of the Missile Defense Project at the Washington-based Center for Strategic and International Studies.
The danger, Karako said, is that the U.S. and Israel could run out of interceptors before they are able to take out the rest of Iran's missile stockpiles and mobile launchers — an objective that has proven “maddeningly difficult.”
Over a month into the war, Trump administration officials continue to refer to the first 72 hours as their point of comparison for claims about Iran’s crippled capacity.
“A good percentage of Iranian missiles, at least half of the arsenal, is stored in very hardened facilities that are not easily reachable with air power,” said Farzin Nadimi, an expert on the Iranian missile program at The Washington Institute. “It looks like the Americans and the Israelis have been underestimating some level of complexity.”
Experts say Iran focuses its attacks to cause economic harm
Contrary to Hegseth’s characterization of the Iranians as “flailing recklessly" by striking civilian and energy infrastructure across the Arabian Peninsula, analysts say Tehran appears to have fine-tuned its timing and targets to maximize damage.
“They have been able to strike targets more efficiently and therefore use fewer missiles to achieve the same result,” Nadimi said.
Iran has increasingly concentrated its firepower on sensitive sites like oil pipelines and water desalination plants across the Persian Gulf in a bid to impose a settlement on the U.S., hitting nearby states like the United Arab Emirates and Kuwait hardest. Last week, Iran fired ballistic missiles and drones at a Saudi air base, wounded more than two dozen U.S. troops and damaging aircraft.
“In this asymmetrical war, the most important thing for Iran is attack the world economy in hopes of coercing the U.S. to stop,” said Assaf Orion, a retired Israeli brigadier general and senior researcher at the Institute for National Security Studies. That has become more important to Iran than attacking Israel, which views this war as existential and won’t be dissuaded, he added.
How long Iran can sustain its current level of retaliation remains unclear, as U.S. and Israeli intelligence on Iran's missile and drone inventory is limited.
Military experts from both countries offer varying estimates on the remaining arsenal, but agree that Iran most likely still has thousands of cheap, locally manufactured drones that it can deploy to menace U.S. allies even if much of its midrange ballistic missile capacity has been destroyed.
“Iran built itself to be able to ride a war like this out,” said Karako. “It has been preparing for this."
___
Toropin reported from Washington.

---

**Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' - Yahoo**

Published: Mon, 30 Mar 2026 23:43:28 GMT

Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic'
By Alexander Cornwell, Trevor Hunnicutt and Asif Shahzad
TEL AVIV/WASHINGTON/ISLAMABAD, March 30 (Reuters) - President Donald Trump warned on Monday that the U.S. would obliterate Iran's energy plants and oil wells if Tehran does not open the Strait of Hormuz, after Tehran described U.S. peace proposals as "unrealistic" and fired waves of missiles at Israel.
Israel's military said two drones from Yemen had also been intercepted on Monday, two days after the Iran-aligned Houthis entered the war by firing missiles at Israel, and that Lebanon's Hezbollah had fired rockets at Israel.
Israeli forces carried out missile strikes on what they called military infrastructure in Tehran and infrastructure used by Iran-backed Hezbollah in Beirut, leaving black smoke hanging over the Lebanese capital.
Turkey's defense ministry said a ballistic missile launched from Iran entered Turkish airspace before being shot down by NATO air and missile defenses deployed in the eastern Mediterranean, the fourth such incident since the start of the war.
Tehran remains defiant in the month-old war, which began with U.S.-Israeli attacks on Iran on February 28 and has spread across the region, killing thousands, disrupting energy supplies and hitting the global economy.
The majority of those reported killed were in Iran and Lebanon, and many were civilians. Iran has effectively blocked the Strait of Hormuz, a narrow waterway that normally carries about a fifth of global oil and liquefied natural gas supplies.
TROOPS DEPLOY AS TALKS CONTINUE
Thousands of soldiers from the U.S. Army's elite 82nd Airborne Division have started arriving in the Middle East, two U.S. officials told Reuters on Monday, part of a reinforcement that would expand Trump's options to include the deployment of forces inside Iranian territory, even as he pursues talks with Tehran.
White House press secretary Karoline Leavitt later said Trump wanted to reach a deal with Tehran before an April 6 deadline he set last week after extending an earlier deadline he had set for Iran to open the Strait of Hormuz. Leavitt said talks with Iran were progressing, adding that what Tehran says publicly differs from what it tells U.S. officials in private.
Iran said earlier on Monday it had received U.S. peace proposals via intermediaries, following talks on Sunday between the foreign ministers of Pakistan, Egypt, Saudi Arabia and Turkey.
Iranian Foreign Ministry spokesperson Esmaeil Baghaei said the proposals were "unrealistic, illogical and excessive".
More in World
"Our position is clear. We are under military aggression. Therefore, all our efforts and strength are focused on defending ourselves," he told a press conference.
Soon after Baghaei's remarks, Trump said in a social media post that the United States was in talks with a "more reasonable regime" to end the war in Iran, but he also issued a new warning over the Strait of Hormuz.
"Great progress has been made but, if for any reason a deal is not shortly reached, which it probably will be, and if the Hormuz Strait is not immediately 'Open for Business,' we will conclude our lovely 'stay' in Iran by blowing up and completely obliterating all of their Electric Generating Plants, Oil Wells and Kharg Island," Trump wrote.
Trump also threatened to attack the desalination plants that supply clean water in Iran.
The national security committee in the Iranian parliament, meanwhile, approved a bill that bans ships from the U.S., Israel and countries that unilaterally sanction Iran, from moving through the Hormuz Strait, according to state media. The bill must still be approved by the full parliament and it was not clear when or if such a vote would take place.
A Pakistani security official, whose country is trying to mediate in the war, said it appeared unlikely there would be direct U.S.-Iran talks this week.
Baghaei also said Iran's parliament was reviewing a possible exit from the Nuclear Non-Proliferation Treaty, which recognizes the right to develop, research, produce and use nuclear energy as long as nuclear weapons are not pursued.
Trump has cited the prevention of Iran from obtaining nuclear weapons as a reason for attacking the country on February 28. Tehran denies it is seeking a nuclear arsenal.
Israeli Prime Minister Benjamin Netanyahu, in an interview with U.S. media outlet Newsmax, declined to give a timeline for achieving his country’s objectives in the war. While he said that "it's definitely beyond the halfway point," he later clarified that he meant in terms of missions, not time.
OIL MARKETS BRACE FOR TURMOIL
The White House said Trump was considering asking Arab nations to pay for the cost of the war. "It's an idea that I know that he has and something that I think you'll hear more from him on," Leavitt said in response to a reporter's question about the idea.
His administration requested an additional $200 billion in funding for the war, which faces stiff opposition in the U.S. Congress, which must approve new spending.
Iran has fired on Arab Gulf states during the conflict and war has been reignited between Israel and Hezbollah in Lebanon. Three members of the U.N. peacekeeping mission in Lebanon (UNIFIL) were killed in two separate incidents in southern Lebanon after a bloody weekend in which Lebanese journalists and medics were killed in Israeli strikes.
Benchmark oil prices extended gains on Monday, with Brent crude futures on course for a record monthly rise.
The Houthis' attacks on Israel raised the prospect that they could target and block a second important shipping route, the Bab el-Mandeb Strait.
The oil market has all but discounted the prospect of a negotiated end to the war and "is bracing for a sharp escalation in military hostilities," said Vandana Hari of oil-market provider Vanda Insights.
The International Monetary Fund warned that war in the Middle East has caused serious disruption to the economies of frontline countries, and is dimming the outlook for many economies that had just started to recover from previous crises.
G7 finance leaders also said they were ready to take "all necessary measures" to safeguard energy market stability and limit broader economic spillovers from recent volatility.
(Reporting by Reuters bureaux; Writing by Stephen Coates, Timothy Heritage, Keith Weir, Simon Lewis and Brad Brooks; Editing by Gareth Jones and Matthew Lewis)

---

**Trump issues fiery new threat against Iran as details of US aviator's rescue emerge - Yahoo**

Published: Sun, 05 Apr 2026 04:10:00 GMT

Trump issues an expletive-filled threat against Iran as details of US aviator's rescue emerge
TEHRAN, Iran (AP) — U.S. President Donald Trump on Sunday made expletive-filled threats against Iran and its infrastructure if it doesn't open the Strait of Hormuz by his Tuesday deadline, after American forces rescued a wounded aviator whose Iran-downed plane fell behind enemy lines.
A defiant Iran struck infrastructure targets in neighboring Gulf Arab countries and threatened to restrict another heavily used waterway, the Bab el-Mandeb Strait off the Arabian Peninsula.
Trump on social media vowed to hit Iran’s power plants and bridges and said the country would be “living in Hell” if the Strait of Hormuz, crucial for global trade, isn’t opened. He ended with “Praise be to Allah.”
Trump has issued such deadlines before but extended them when mediators have claimed progress toward ending the war, which has killed thousands, shaken global markets and spiked fuel prices in just over five weeks.
“It seems Trump has become a phenomenon that neither Iranians nor Americans are able to fully analyze,” Iranian Culture Minister Sayed Reza Salihi-Amiri told visiting Associated Press journalists in an interview in Tehran, adding that the U.S. president “constantly shifts between contradictory positions.”
Both sides have threatened and hit civilian targets like oil fields and desalination plants that provide drinking water. Iran’s U.N. mission called Trump’s threat “clear evidence of intent to commit war crime.”
Iran’s joint military command warned of stepped-up attacks on regional oil and civilian infrastructure if the U.S. and Israel attack such targets there, according to state television.
The laws of armed conflict allow attacks on civilian infrastructure only if the military advantage outweighs the civilian harm, legal scholars say. It’s considered a high bar to clear, and causing excessive suffering to civilians can constitute a war crime.
The US describes a dramatic rescue
An intense search followed Friday's crash of the F-15E Strike Eagle, while Iran promised a reward for the “enemy pilot.” It was the first known American aircraft to crash in Iranian territory since the U.S. and Israel launched the war on Feb. 28.
Trump said that the service member was “seriously wounded and really brave” and rescued from “deep inside the mountains" in an operation involving dozens of armed aircraft. He said a second crew member was rescued in “broad daylight” within hours of the crash.
A senior U.S. administration official said that before locating the second aviator, the CIA spread word inside Iran that U.S. forces had found him and were moving him out, creating confusion for Iranians. The official spoke on the condition of anonymity to discuss details not yet made public.
Iran also shot down another U.S. military plane Friday, demonstrating the perils of the bombing campaign and the ability of Iran's degraded military to hit back. Neither the status of the A-10 attack aircraft's crew nor where it crashed is known.
More in World
On Sunday, Iran’s state television aired a video showing what it claimed were parts of U.S. aircraft — a transport plane and two helicopters — shot down by Iranian forces during the rescue operation.
However, a regional intelligence official briefed on the mission told the AP that the U.S. military blew up two transport planes because of a technical malfunction and brought in additional aircraft to complete the rescue. The official spoke on condition of anonymity to discuss the covert mission.
Iran’s joint military command later said the U.S. bombarded its own aircraft to “prevent embarrassment for President Trump."
Two Black Hawk helicopters were hit but navigated to safe airspace, according to a person familiar with the situation who spoke on condition of anonymity to discuss the sensitive information.
Diplomatic efforts continue
Trump's deadline centers on alarm over Iran's grip on the Strait of Hormuz, critical for global shipments of oil and gas from the Persian Gulf as well as humanitarian supplies. Some ships have paid Iran for passage.
An Iranian presidential spokesperson, Seyyed Mohammad Mehdi Tabatabaei, said on social media that the strait can reopen only if some transit revenues compensate Iran for war damages.
A top Iranian adviser, Ali Akbar Velayati, warned on social media that Tehran also could disrupt trade on the Bab el-Mandeb, a key chokepoint to and from the Red Sea.
Diplomatic efforts continued. Oman's Foreign Ministry said that deputy foreign ministers and experts from Iran and Oman met to discuss proposals to ensure “smooth transit” through the strait.
Egypt said that Foreign Minister Badr Abdelatty had spoken with U.S. envoy Steve Witkoff and Iranian Foreign Minister Abbas Araghchi, and with Turkish and Pakistani counterparts. Russia said that Araghchi also spoke with Russian Foreign Minister Sergey Lavrov.
Bahrain urged the U.N. Security Council to act on its draft proposal with language authorizing defensive action to ensure safe passage through the strait.
Airstrikes hit Iran
An airstrike early Monday struck a residential building near Eslamshar, southwest of Tehran, killing at least 13 people, the semiofficial Fars news agency and Nour News reported.
Airstrikes also damaged buildings at the Sharif University of Technology in Tehran as well as a natural gas distribution site next to the campus, Iranian media reported. It wasn’t immediately clear what was targeted at the university campus, which has switched to online classes because of the war.
Elsewhere in Iran, an airstrike killed at least five people in a residential area of Qom, the state-run IRAN daily newspaper said in an online message. Qom is a Shiite seminary city just south of Tehran.
It wasn't clear why the buildings were struck. Neither Israel nor the United States claimed the strikes early Monday
In the United Arab Emirates, authorities said one Nepali and three Pakistanis were hurt in fires caused by debris from the interception of an Iranian projectile at Khor Fakkan port, and interception debris caused fires at a petrochemical plant in Ruwais, halting operations.
In Kuwait, Iranian drone attacks caused significant damage to power plants and a petrochemical plant. They also put a water desalination station out of service, according to the Ministry of Electricity.
In Bahrain, a drone attack caused a fire at a national oil company storage facility and a state-run petrochemical plant, the kingdom’s official news agency said.
In Israel, rescue authorities searched for three people in the northern city of Haifa after an apartment building was hit. It wasn't immediately clear what struck it.
More than 1,900 people have been killed in Iran since the war began, but its government has not updated the toll for days.
In Lebanon, whose health ministry said an Israeli strike without warning killed four people in Beirut, more than 1,400 people have been killed and more than 1 million people have been displaced. Eleven Israeli soldiers have died there while targeting Iranian-backed Hezbollah militants.
In Gulf Arab states and the occupied West Bank, more than two dozen people have died, while 19 have been reported dead in Israel and 13 U.S. service members have been killed.
___
Lee and Toropin reported from Washington, Metz from Jerusalem and Magdy from Cairo. Associated Press writers Jon Gambrell in Dubai, United Arab Emirates; Lisa Mascaro and Seung Min Kim in Washington; Munir Ahmed in Islamabad; Farnoush Amiri in New York; and Christopher Weber in Los Angeles contributed to this report.

...

---

**Videos show fire and explosions in Isfahan, Iran, as the conflict continues - NBC News**

Published: Tue, 31 Mar 2026 08:23:09 GMT

IE 11 is not supported. For an optimal experience visit our site on another browser.
Exhaustive search underway in Iran to rescue downed F-15 airman
02:20Surcharges hit consumers in economic fallout from war with Iran
02:02Second U.S. aircraft struck by Iranian fire after downed jet, U.S. official says
04:17Artemis II astronauts describe their journey as they rocket toward the moon
01:11Multiple airstrikes hit Iranian missile sites in Isfahan
00:24Trump hints at timeline for war with Iran during address to nation
06:38Trump says U.S. strikes will send Iran back to 'Stone Ages'
01:08Palestinians protest Israel’s new death penalty law
01:27Trump suggests war with Iran will end within two weeks
02:27Israeli airstrike destroys Beirut building
00:43Hegseth says U.S. made 200 dynamic strikes overnight, but other Iran questions are still unanswered
01:52Video from Iranian state media shows funeral procession for IRGC naval chief
00:36Now Playing
Videos show fire and explosions in Isfahan, Iran, as the conflict continues
00:28UP NEXT
President Trump claims 'regime change' has happened in Iran
00:32Energy disruptions from Iran war trigger sprawling oil shortages across Asia
03:39Remains of French musketeer d'Artagnan may have been found in Dutch church
00:37U.S. stocks suffer biggest loss since the war with Iran started
06:41New IOC policy bans transgender women from Olympics
02:32Judge says he won’t dismiss Nicolás Maduro’s case over legal fees dispute
08:59Bus falls into river while boarding ferry in Bangladesh, leaving 24 dead
00:21
Videos show fire and explosions in Isfahan, Iran, as the conflict continues
00:28Videos geolocated by NBC News show intense overnight explosions sending a fireball into the sky above Iran's central city of Isfahan.March 31, 2026
UP NEXT
Exhaustive search underway in Iran to rescue downed F-15 airman
02:20Surcharges hit consumers in economic fallout from war with Iran
02:02Second U.S. aircraft struck by Iranian fire after downed jet, U.S. official says
04:17Artemis II astronauts describe their journey as they rocket toward the moon
01:11Multiple airstrikes hit Iranian missile sites in Isfahan
00:24Trump hints at timeline for war with Iran during address to nation
06:38
NBC News Channel
Meet the Press
Meet the Press
Play All

---

**'This war is not going to end anytime soon,' says fmr. Pentagon spokesperson on Iran conflict - CNN**

Published: Fri, 03 Apr 2026 12:14:23 GMT

Video Ad Feedback
'This war is not going to end anytime soon,' says fmr. Pentagon spokesperson on Iran conflict
As Trump threatens to continue bombing Iran and send the country "back to the Stone Age", Christiane Amanpour speaks to former Pentagon spokesperson and White House official John Kirby about what viable exit strategy still exists for Washington.
15:50
• Source:
CNN
Amanpour
15 videos
Video Ad Feedback
'This war is not going to end anytime soon,' says fmr. Pentagon spokesperson on Iran conflict
Video Ad Feedback
The US is a rogue nation, says fmr. White House expert on Iran
Video Ad Feedback
The true story that led Adrien Brody to his Broadway debut in 'The Fear of 13'
Video Ad Feedback
With Iran, Trump's 'usual playbook is not working out,' says scholar
Video Ad Feedback
'That Night': Documenting the plight of Iranian political prisoners
Video Ad Feedback
Could the war in Iran push Tehran closer to a nuclear weapon? A former US official weighs in
Video Ad Feedback
Ukraine's calls for help falling on deaf ears, former foreign minister says
Video Ad Feedback
‘Cubans just want normal lives,’ says journalist on US-Cuba tensions
Video Ad Feedback
'Lebanese people are being held captive in this conflict,' says Lebanese political activist
Video Ad Feedback
European allies shocked by Hegseth criticisms, says former US Amb. to NATO
Video Ad Feedback
If Trump puts troops on the ground ‘we will be trapped,’ warns fmr. CIA chief
Video Ad Feedback
If Iranian regime survives, it will go after Iranian people, brother of imprisoned Nobel laureate says
Video Ad Feedback
Why the Strait of Hormuz crisis is making the case for renewables
Video Ad Feedback
‘We’ll be paying for this war for years to come’: UN humanitarian chief Tom Fletcher on war in Iran and Lebanon
Video Ad Feedback
War may leave Iranian people ‘in an even worse position,’ says US congresswoman

---

### Entire History Back to LLM

The entire context must be sent back to LLM to continue.

**Response API**: build the next `input` by appending `response.output` items + tool results:

```python
# input = original_input + response.output + [function_call_output, ...]
input_with_tools = input_prompt + list(response.output) + tool_outputs
```

| Layer | Chat Completion | Response API |
|---|---|---|
| Original prompt | `messages` list | `input` list |
| LLM decision | `choices[0].message` dict (with `tool_calls`) | `response.output` items (`type=="function_call"`) |
| Tool results | `{"role":"tool","tool_call_id":...,"content":...}` | `{"type":"function_call_output","call_id":...,"output":...}` |

* [Incorporating results into response](https://developers.openai.com/api/docs/guides/function-calling#incorporating-results-into-response)

> After appending the results to your input, send them back to get a final response:
> ```python
> response = client.responses.create(
>     model="gpt-4.1",
>     input=input_messages,
>     tools=tools,
> )
> ```


In [22]:
# Build the next input by appending the full response.output + tool results.
# response.output contains both function_call items and any assistant text items.
# Ref: https://docs.litellm.ai/docs/response_api
#      https://developers.openai.com/api/docs/guides/function-calling#incorporating-results-into-response
input_with_tool_calls: List[Any] = input_prompt + list(response.output) + tool_outputs

response = litellm.responses(
    model=MODEL,
    instructions=SYSTEM_INSTRUCTIONS,
    input=input_with_tool_calls,
    temperature=0,
    max_output_tokens=2048,
    text={
        "format": {
            # Structured Output: force the model to emit valid JSON matching the schema.
            "type": "json_schema",
            "name": "ResponseStructuredFormat",
            "strict": True,
            "schema": ResponseStructuredFormat.model_json_schema()
        }
    },
    stream=False
)

In [23]:
# litellm has no .output_text shortcut — extract text from the first completed message item.
# Compare with Chat Completion (deprecated): response.choices[0].message.content
message = next(item for item in response.output if item.type == "message")
print(message.content[0].text)

{"news":["Trump issues fiery new threat against Iran as details of US aviator's rescue emerge (Yahoo)","US vows to target more Iranian infrastructure as nations seek to open Hormuz (Reuters)","'However Long It Takes': USS Bush Gets Underway as Iran Conflict Rages (Military.com)","US Ground Forces Arrive in Middle East as Iran Conflict Escalates (Military.com)","Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' (Yahoo)","Iran remains a stubborn foe after absorbing massive US-Israeli attacks (Yahoo/AP)"],"uri":["https://www.yahoo.com/news/articles/us-member-missing-iran-shot-041018922.html","https://www.reuters.com/world/middle-east/us-vows-target-more-iranian-infrastructure-nations-seek-open-hormuz-2026-04-03/","https://www.military.com/daily-news/2026/04/01/however-long-it-takes-uss-bush-gets-underway-iran-conflict-rages.html","https://www.military.com/benefits/2026/03/30/us-ground-forces-arrive-middle-east-iran-conflict-escalates.html","https://www.yahoo.

In [24]:
# litellm response is dict-like — use json.dumps instead of .model_dump()
print(json.dumps(response, indent=2, default=str))

"id='resp_bGl0ZWxsbTpjdXN0b21fbGxtX3Byb3ZpZGVyOm9wZW5haTttb2RlbF9pZDpOb25lO3Jlc3BvbnNlX2lkOnJlc3BfMDgzZjJlZTlmZTA2Yzc0NDAwNjlkMzI3Yzk1ZjE4ODFhMzk5ZWFkZDkzOTY4MjBiMmI=' created_at=1775445961 error=None incomplete_details=None instructions='\\nYou are a helpful assistant. Use tools when needed.\\nVerify the relevance of retrieved results before using them, and apply an intelligent\\nintent filter so you keep only the news items that align with the user inquiry.\\nOnly report items that are relevant to the user request and supported by the tool output.\\nIf the search results are noisy or insufficient, say that and do not infer missing facts.\\nRespnse follows the given Python JSON schema where:\\n1. news field is the news articies as list.\\n2. uri field is the URI of the news article found.\\n' metadata={} model='gpt-5.2-2025-12-11' object='response' output=[ResponseOutputMessage(id='msg_083f2ee9fe06c7440069d327ca0a3081a3b0058fc8dff609de', content=[ResponseOutputText(annotations=[], tex

In [25]:
json.loads(response.output[0].content[0].text)['news']

["Trump issues fiery new threat against Iran as details of US aviator's rescue emerge (Yahoo)",
 'US vows to target more Iranian infrastructure as nations seek to open Hormuz (Reuters)',
 "'However Long It Takes': USS Bush Gets Underway as Iran Conflict Rages (Military.com)",
 'US Ground Forces Arrive in Middle East as Iran Conflict Escalates (Military.com)',
 "Trump issues new warning to Tehran, Iran calls US peace proposals 'unrealistic' (Yahoo)",
 'Iran remains a stubborn foe after absorbing massive US-Israeli attacks (Yahoo/AP)']

In [26]:
del tool_calls, tool_outputs, input_with_tool_calls, response

---